In [1]:
"""
07 - Robustness Check: Additional Event Days and Windows
--------------------------------------------------------
Re-tests abnormal returns on the announcement day and the
following day, and over post-announcement windows.

Event days    : (0), (+1)
Event windows : (0, +1), (0, +5), (0, +10)

Test method
For AR / CAR   : Cross-sectional t
For SAR / CSAR : Standardized cross-sectional t

Data source : clean_event_data.csv
"""

import pandas as pd
import numpy as np
from scipy import stats

df = pd.read_csv("../data/clean_event_data.csv")
N = df["firm"].nunique()

periods = {
    "(0)": (0, 0),
    "(+1)": (1, 1),
    "(0, +1)": (0, 1),
    "(0, +5)": (0, 5),
    "(0, +10)": (0, 10),
}

def star(t):
    p = 2 * stats.norm.sf(abs(t))
    return "***" if p < 0.01 else "**" if p < 0.05 else "*" if p < 0.1 else ""

rows_a, rows_b = [], []
for name, (start, end) in periods.items():
    temp = df[(df["Tau"] >= start) & (df["Tau"] <= end)]

    # Panel A: AR / CAR
    car = temp.groupby("firm")["AR"].sum()
    mean_car = car.mean()
    t_cs = mean_car / (car.std(ddof=1) / np.sqrt(N))
    rows_a.append([name, f"{mean_car:.4f}",
                   f"{t_cs:.4f}{star(t_cs)}"])

    # Panel B: SAR / CSAR
    csar = temp.groupby("firm")["SAR"].sum()
    mean_csar = csar.mean()
    t_std_cs = mean_csar / (csar.std(ddof=1) / np.sqrt(N))
    rows_b.append([name, f"{mean_csar:.4f}",
                   f"{t_std_cs:.4f}{star(t_std_cs)}"])

panel_a = pd.DataFrame(rows_a, columns=["Period", "AR / CAR (%)",
                                        "Cross-sectional t"])
panel_b = pd.DataFrame(rows_b, columns=["Period", "SAR / CSAR",
                                        "Std. cross-sectional t"])

print("Panel A: AR and CAR")
print(panel_a.to_string(index=False))
print("\nPanel B: SAR and CSAR")
print(panel_b.to_string(index=False))
print("\n* p < 0.10, ** p < 0.05, *** p < 0.01 (two-tailed, normal)")

Panel A: AR and CAR
  Period AR / CAR (%) Cross-sectional t
     (0)       0.0803            0.1465
    (+1)       1.2052            0.8117
 (0, +1)       1.2855            0.8527
 (0, +5)      -1.7014           -0.6792
(0, +10)      -2.1902           -0.7962

Panel B: SAR and CSAR
  Period SAR / CSAR Std. cross-sectional t
     (0)    -0.0181                -0.0790
    (+1)     0.4471                 0.6206
 (0, +1)     0.4290                 0.6133
 (0, +5)    -0.6537                -0.4863
(0, +10)    -0.7865                -0.5537

* p < 0.10, ** p < 0.05, *** p < 0.01 (two-tailed, normal)
